# 04 — Dataset Construction

This notebook combines Polymarket, Kalshi, and Deribit inputs into the final empirical datasets used in the thesis.

The main output is a set of matched datasets containing prediction-market probabilities, Deribit-implied probability proxies, and cross-market mispricing measures.

In [2]:
# ============================================================
# Imports and project configuration
# ============================================================

import os
import math

import numpy as np
import pandas as pd

from scipy.stats import norm

from importlib import reload
import config
reload(config)

from config import *

print("SAMPLE_START:", SAMPLE_START)
print("SAMPLE_END:", SAMPLE_END)
print("FINAL_DIR:", FINAL_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("TABLES_DIR:", TABLES_DIR)

for folder in [FINAL_DIR, PROCESSED_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SAMPLE_START: 2024-01-01
SAMPLE_END: 2026-06-04
FINAL_DIR: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final
PROCESSED_DIR: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed
TABLES_DIR: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables


In [61]:
# ============================================================
# Load final cleaned datasets
# ============================================================

polymarket_path = (
    FINAL_DIR / "polymarket_btc_prices_analysis.csv"
)

kalshi_path = (
    FINAL_DIR / "kalshi_kxbtc_first_trades_timestamped_30m.csv"
)

deribit_path = (
    FINAL_DIR / "deribit_btc_daily_inputs.csv"
)

kalshi_intraday_benchmark_path = (
    FINAL_DIR / "deribit_kalshi_intraday_inputs.csv"
)


df_poly = pd.read_csv(
    polymarket_path,
    low_memory=False,
)

df_kalshi = pd.read_csv(
    kalshi_path,
    low_memory=False,
)

df_deribit = pd.read_csv(
    deribit_path,
    low_memory=False,
)

df_kalshi_benchmark = pd.read_csv(
    kalshi_intraday_benchmark_path,
    low_memory=False,
)


# ------------------------------------------------------------
# Parse Polymarket timestamps
# ------------------------------------------------------------

for col in ["datetime", "end_date"]:
    df_poly[col] = pd.to_datetime(
        df_poly[col],
        utc=True,
        errors="coerce",
    )

df_poly["date"] = (
    df_poly["datetime"].dt.normalize()
)


# ------------------------------------------------------------
# Parse Kalshi timestamps
# ------------------------------------------------------------

for col in [
    "observation_time",
    "open_time",
    "close_time",
]:
    df_kalshi[col] = pd.to_datetime(
        df_kalshi[col],
        utc=True,
        errors="coerce",
    )

df_kalshi["date"] = (
    df_kalshi["observation_time"].dt.normalize()
)


# ------------------------------------------------------------
# Parse daily Deribit dates
# ------------------------------------------------------------

df_deribit["date"] = (
    pd.to_datetime(
        df_deribit["date"],
        utc=True,
        errors="coerce",
    )
    .dt.normalize()
)


# ------------------------------------------------------------
# Parse Kalshi benchmark timestamps
# ------------------------------------------------------------

for col in [
    "observation_time",
    "spot_available_at",
    "benchmark_source_date",
    "benchmark_available_at",
]:
    df_kalshi_benchmark[col] = pd.to_datetime(
        df_kalshi_benchmark[col],
        utc=True,
        errors="coerce",
    )


# ------------------------------------------------------------
# Validate input datasets
# ------------------------------------------------------------

assert len(df_kalshi) == 3_135
assert len(df_kalshi_benchmark) == 3_135

assert df_kalshi["ticker"].is_unique
assert df_kalshi_benchmark["ticker"].is_unique

assert set(df_kalshi["ticker"]) == set(
    df_kalshi_benchmark["ticker"]
)

assert df_kalshi["observation_time"].notna().all()
assert df_kalshi_benchmark[
    ["spot", "dvol", "sigma"]
].notna().all().all()

assert (
    df_kalshi_benchmark["spot_available_at"]
    <= df_kalshi_benchmark["observation_time"]
).all()

assert (
    df_kalshi_benchmark["benchmark_available_at"]
    <= df_kalshi_benchmark["observation_time"]
).all()

# Verify exact timestamp alignment by ticker
kalshi_timing_check = df_kalshi[
    ["ticker", "observation_time"]
].merge(
    df_kalshi_benchmark[
        ["ticker", "observation_time"]
    ],
    on="ticker",
    how="inner",
    suffixes=("_market", "_benchmark"),
    validate="one_to_one",
)

assert len(kalshi_timing_check) == len(df_kalshi)

assert (
    kalshi_timing_check["observation_time_market"]
    == kalshi_timing_check["observation_time_benchmark"]
).all()

print("Kalshi input validation passed.")

# ------------------------------------------------------------
# Output diagnostics
# ------------------------------------------------------------

print("=" * 70)
print("Loaded final datasets")
print("=" * 70)

print("Polymarket:")
print(f"  Rows: {len(df_poly):,}")
print(f"  Markets: {df_poly['market_id'].nunique():,}")
print(
    f"  Date range: "
    f"{df_poly['date'].min()} -> "
    f"{df_poly['date'].max()}"
)

print("\nKalshi timestamped:")
print(f"  Rows: {len(df_kalshi):,}")
print(f"  Events: {df_kalshi['event_ticker'].nunique():,}")
print(f"  Tickers: {df_kalshi['ticker'].nunique():,}")
print(
    f"  Observation range: "
    f"{df_kalshi['observation_time'].min()} -> "
    f"{df_kalshi['observation_time'].max()}"
)

print("\nDeribit daily:")
print(f"  Rows: {len(df_deribit):,}")
print(
    f"  Date range: "
    f"{df_deribit['date'].min()} -> "
    f"{df_deribit['date'].max()}"
)

print("\nKalshi intraday benchmark:")
print(f"  Rows: {len(df_kalshi_benchmark):,}")
print(
    f"  Events: "
    f"{df_kalshi_benchmark['event_ticker'].nunique():,}"
)
print(
    "  Missing inputs:",
    df_kalshi_benchmark[
        ["spot", "dvol", "sigma"]
    ].isna().sum().sum(),
)
print(
    "  Maximum spot age in seconds:",
    round(df_kalshi_benchmark["spot_age_seconds"].max(), 3),
)
print(
    "  Maximum DVOL age in hours:",
    round(df_kalshi_benchmark["dvol_age_hours"].max(), 3),
)

Kalshi input validation passed.
Loaded final datasets
Polymarket:
  Rows: 30,060
  Markets: 3,432
  Date range: 2024-03-09 00:00:00+00:00 -> 2026-06-03 00:00:00+00:00

Kalshi timestamped:
  Rows: 3,135
  Events: 485
  Tickers: 3,135
  Observation range: 2025-02-04 04:00:28.774289+00:00 -> 2026-06-04 03:27:16.183834+00:00

Deribit daily:
  Rows: 886
  Date range: 2024-01-01 00:00:00+00:00 -> 2026-06-04 00:00:00+00:00

Kalshi intraday benchmark:
  Rows: 3,135
  Events: 485
  Missing inputs: 0
  Maximum spot age in seconds: 59.983
  Maximum DVOL age in hours: 4.499


In [63]:
# ============================================================
# Standardize and validate required columns
# ============================================================

# ------------------------------------------------------------
# Standardize Polymarket probability
# ------------------------------------------------------------

if "prob_yes" not in df_poly.columns:
    if "price" in df_poly.columns:
        df_poly["prob_yes"] = pd.to_numeric(
            df_poly["price"],
            errors="coerce",
        )
    elif "prob_market" in df_poly.columns:
        df_poly["prob_yes"] = pd.to_numeric(
            df_poly["prob_market"],
            errors="coerce",
        )
    else:
        raise ValueError(
            "Polymarket probability column not found."
        )


# ------------------------------------------------------------
# Ensure numeric input columns
# ------------------------------------------------------------

for col in [
    "prob_kalshi",
    "floor_strike",
    "cap_strike",
    "minutes_after_open",
    "minutes_to_close",
]:
    df_kalshi[col] = pd.to_numeric(
        df_kalshi[col],
        errors="coerce",
    )

for col in ["spot", "dvol", "sigma"]:
    df_deribit[col] = pd.to_numeric(
        df_deribit[col],
        errors="coerce",
    )

for col in [
    "spot",
    "dvol",
    "sigma",
    "spot_age_seconds",
    "dvol_age_hours",
]:
    df_kalshi_benchmark[col] = pd.to_numeric(
        df_kalshi_benchmark[col],
        errors="coerce",
    )


# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

required_poly_cols = [
    "datetime",
    "end_date",
    "prob_yes",
    "market_id",
    "bet_type",
    "strike_lo",
    "strike_hi",
]

required_kalshi_cols = [
    "ticker",
    "event_ticker",
    "observation_time",
    "open_time",
    "close_time",
    "strike_type",
    "floor_strike",
    "cap_strike",
    "prob_kalshi",
    "price_source",
    "minutes_after_open",
    "minutes_to_close",
    "first_trade_size",
]

required_deribit_cols = [
    "date",
    "spot",
    "dvol",
    "sigma",
]

required_kalshi_benchmark_cols = [
    "ticker",
    "event_ticker",
    "observation_time",
    "spot_available_at",
    "spot_age_seconds",
    "spot",
    "benchmark_source_date",
    "benchmark_available_at",
    "dvol_age_hours",
    "dvol",
    "sigma",
]


# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

for name, frame, required_cols in [
    ("Polymarket", df_poly, required_poly_cols),
    ("Kalshi timestamped", df_kalshi, required_kalshi_cols),
    ("Deribit daily", df_deribit, required_deribit_cols),
    (
        "Kalshi intraday benchmark",
        df_kalshi_benchmark,
        required_kalshi_benchmark_cols,
    ),
]:
    missing_cols = [
        col for col in required_cols
        if col not in frame.columns
    ]

    print("=" * 70)
    print(name)
    print("Missing required columns:", missing_cols)

    if missing_cols:
        raise ValueError(
            f"{name} is missing required columns: "
            f"{missing_cols}"
        )


# ------------------------------------------------------------
# Validate input values
# ------------------------------------------------------------

assert df_poly["prob_yes"].between(
    0,
    1,
    inclusive="both",
).all()

assert df_kalshi["prob_kalshi"].between(
    0,
    1,
    inclusive="both",
).all()

assert df_kalshi["minutes_after_open"].between(
    0,
    30,
    inclusive="both",
).all()

assert (df_kalshi["minutes_to_close"] > 0).all()

assert (df_kalshi_benchmark["spot"] > 0).all()
assert (df_kalshi_benchmark["dvol"] > 0).all()
assert (df_kalshi_benchmark["sigma"] > 0).all()

print("=" * 70)
print("All required columns and input values are valid.")

Polymarket
Missing required columns: []
Kalshi timestamped
Missing required columns: []
Deribit daily
Missing required columns: []
Kalshi intraday benchmark
Missing required columns: []
All required columns and input values are valid.


In [65]:
# ============================================================
# Build temporally available daily benchmark
# ============================================================

df_deribit_available = (
    df_deribit
    .sort_values("date")
    .copy()
)

# Date represented by the original daily close.
df_deribit_available["benchmark_source_date"] = (
    df_deribit_available["date"]
)

# A daily close becomes safely available from the next UTC day.
df_deribit_available["benchmark_available_at"] = (
    df_deribit_available["benchmark_source_date"]
    + pd.Timedelta(days=1)
)

# This is the date used to match prediction-market observations.
df_deribit_available["date"] = (
    df_deribit_available["benchmark_available_at"].dt.normalize()
)

benchmark_cols = [
    "date",
    "benchmark_source_date",
    "benchmark_available_at",
    "spot",
    "dvol",
    "sigma",
]

df_deribit_available = df_deribit_available[benchmark_cols].copy()

# Verify the benchmark availability convention.
expected_availability = (
    df_deribit_available["benchmark_source_date"]
    + pd.Timedelta(days=1)
)

if not (
    df_deribit_available["benchmark_available_at"]
    == expected_availability
).all():
    raise ValueError(
        "Invalid benchmark availability-date construction."
    )

if df_deribit_available["date"].duplicated().any():
    raise ValueError(
        "Duplicate benchmark availability dates detected."
    )
# ------------------------------------------------------------
# Preliminary Polymarket timing check
# ------------------------------------------------------------

df_poly_timing_check = df_poly[["datetime"]].copy()

df_poly_timing_check["date"] = (
    df_poly_timing_check["datetime"].dt.normalize()
)

df_poly_timing_check = df_poly_timing_check.merge(
    df_deribit_available,
    on="date",
    how="left",
)

timing_violation = (
    df_poly_timing_check["benchmark_available_at"]
    > df_poly_timing_check["datetime"]
)

print("=" * 70)
print("Temporally available Deribit benchmark")
print("=" * 70)

print("Benchmark rows:", len(df_deribit_available))

print("\nSource-date range:")
print(df_deribit_available["benchmark_source_date"].min())
print(df_deribit_available["benchmark_source_date"].max())

print("\nAvailability-date range:")
print(df_deribit_available["date"].min())
print(df_deribit_available["date"].max())

print("\nPolymarket timing check:")
print(f"  Observations: {len(df_poly_timing_check):,}")
print(
    "  Missing benchmark:",
    df_poly_timing_check["spot"].isna().sum()
)
print(
    "  Benchmark available after observation:",
    timing_violation.fillna(False).sum()
)

display(df_deribit_available.head())

Temporally available Deribit benchmark
Benchmark rows: 886

Source-date range:
2024-01-01 00:00:00+00:00
2026-06-04 00:00:00+00:00

Availability-date range:
2024-01-02 00:00:00+00:00
2026-06-05 00:00:00+00:00

Polymarket timing check:
  Observations: 30,060
  Missing benchmark: 0
  Benchmark available after observation: 0


,date,benchmark_source_date,benchmark_available_at,spot,dvol,sigma
0,2024-01-02 00:00:00+00:00,2024-01-01 00:00:00+00:00,2024-01-02 00:00:00+00:00,44179.55,66.81,0.6681
1,2024-01-03 00:00:00+00:00,2024-01-02 00:00:00+00:00,2024-01-03 00:00:00+00:00,44946.91,63.75,0.6375
2,2024-01-04 00:00:00+00:00,2024-01-03 00:00:00+00:00,2024-01-04 00:00:00+00:00,42845.23,65.17,0.6517
3,2024-01-05 00:00:00+00:00,2024-01-04 00:00:00+00:00,2024-01-05 00:00:00+00:00,44151.10,65.34,0.6534
4,2024-01-06 00:00:00+00:00,2024-01-05 00:00:00+00:00,2024-01-06 00:00:00+00:00,44145.11,67.64,0.6764


In [67]:
# ============================================================
# Black-Scholes probability helper functions
# ============================================================

RISK_FREE_RATE = 0.0
MIN_TAU = 1 / (365 * 24 * 60)
MIN_SIGMA = 1e-6


def bs_prob_above(spot, strike, tau, sigma, r=RISK_FREE_RATE):
    if pd.isna(spot) or pd.isna(strike) or pd.isna(tau) or pd.isna(sigma):
        return np.nan

    if spot <= 0 or strike <= 0 or tau <= 0 or sigma <= 0:
        return np.nan

    tau = max(float(tau), MIN_TAU)
    sigma = max(float(sigma), MIN_SIGMA)

    d2 = (
        np.log(float(spot) / float(strike)) +
        (r - 0.5 * sigma ** 2) * tau
    ) / (sigma * np.sqrt(tau))

    return norm.cdf(d2)


def bs_prob_below(spot, strike, tau, sigma, r=RISK_FREE_RATE):
    prob_above = bs_prob_above(spot, strike, tau, sigma, r=r)

    if pd.isna(prob_above):
        return np.nan

    return 1 - prob_above


def bs_prob_between(spot, strike_low, strike_high, tau, sigma, r=RISK_FREE_RATE):
    if pd.isna(strike_low) or pd.isna(strike_high):
        return np.nan

    low = min(float(strike_low), float(strike_high))
    high = max(float(strike_low), float(strike_high))

    prob_below_high = bs_prob_below(spot, high, tau, sigma, r=r)
    prob_below_low = bs_prob_below(spot, low, tau, sigma, r=r)

    if pd.isna(prob_below_high) or pd.isna(prob_below_low):
        return np.nan

    return max(prob_below_high - prob_below_low, 0)


def compute_deribit_probability(row, source):
    spot = row.get("spot")
    sigma = row.get("sigma")
    tau = row.get("tau")

    if source == "polymarket":
        bet_type = row.get("bet_type")
        strike_lo = row.get("strike_lo")
        strike_hi = row.get("strike_hi")

        if bet_type in ["above", "reach"]:
            return bs_prob_above(spot, strike_lo, tau, sigma)

        if bet_type == "dip":
            return bs_prob_below(spot, strike_lo, tau, sigma)

        if bet_type == "range":
            return bs_prob_between(spot, strike_lo, strike_hi, tau, sigma)

        return np.nan

    if source == "kalshi":
        strike_type = row.get("strike_type")
        floor_strike = row.get("floor_strike")
        cap_strike = row.get("cap_strike")

        if strike_type == "greater":
            return bs_prob_above(spot, floor_strike, tau, sigma)

        if strike_type == "less":
            return bs_prob_below(spot, cap_strike, tau, sigma)

        if strike_type == "between":
            return bs_prob_between(spot, floor_strike, cap_strike, tau, sigma)

        return np.nan

    return np.nan

In [69]:
# ============================================================
# Construct Polymarket-Deribit matched dataset
# ============================================================

df_poly_m = df_poly.copy()

df_poly_m = df_poly_m.merge(
    df_deribit_available[
        [
            "date",
            "benchmark_source_date",
            "benchmark_available_at",
            "spot",
            "sigma",
            "dvol",
        ]
    ],
    on="date",
    how="left",
)

# The benchmark must exist and must already be available
# when the Polymarket price is observed.
required_benchmark_cols = [
    "spot",
    "sigma",
    "dvol",
    "benchmark_source_date",
    "benchmark_available_at",
]

if df_poly_m[required_benchmark_cols].isna().any().any():
    missing_benchmark_inputs = (
        df_poly_m[required_benchmark_cols]
        .isna()
        .sum()
    )

    raise ValueError(
        "Some Polymarket observations have incomplete "
        f"benchmark inputs:\n{missing_benchmark_inputs}"
    )

if not (
    df_poly_m["benchmark_available_at"]
    <= df_poly_m["datetime"]
).all():
    raise ValueError(
        "Look-ahead detected: benchmark available after Polymarket observation."
    )

df_poly_m["tau_raw"] = (
    (df_poly_m["end_date"] - df_poly_m["datetime"])
    .dt.total_seconds() / (365 * 24 * 60 * 60)
)

df_poly_m["valid_time_to_expiry"] = df_poly_m["tau_raw"] > 0

df_poly_m["tau"] = np.nan
df_poly_m.loc[df_poly_m["valid_time_to_expiry"], "tau"] = (
    df_poly_m.loc[df_poly_m["valid_time_to_expiry"], "tau_raw"]
    .clip(lower=MIN_TAU)
)

df_poly_m["prob_market"] = df_poly_m["prob_yes"]

df_poly_m["prob_deribit"] = np.nan

valid_time_mask = df_poly_m["valid_time_to_expiry"]

df_poly_m.loc[valid_time_mask, "prob_deribit"] = df_poly_m.loc[
    valid_time_mask
].apply(
    lambda row: compute_deribit_probability(row, source="polymarket"),
    axis=1,
)

df_poly_m["mispricing"] = df_poly_m["prob_market"] - df_poly_m["prob_deribit"]

df_poly_m["valid_match"] = (
    df_poly_m["valid_time_to_expiry"] &
    df_poly_m["prob_market"].between(0, 1, inclusive="both") &
    df_poly_m["prob_deribit"].between(0, 1, inclusive="both") &
    df_poly_m["spot"].notna() &
    df_poly_m["sigma"].notna() &
    df_poly_m["tau"].notna()
)

df_poly_matched = df_poly_m[df_poly_m["valid_match"]].copy()

print("=" * 70)
print("Polymarket-Deribit matched dataset")
print("=" * 70)

print(f"Input rows: {len(df_poly_m):,}")
print(f"Rows with tau_raw <= 0 excluded: {(~df_poly_m['valid_time_to_expiry']).sum():,}")
print(f"Matched rows: {len(df_poly_matched):,}")
print(f"Markets matched: {df_poly_matched['market_id'].nunique():,}")

print("\nMissing inputs:")
print(df_poly_m[["spot", "sigma", "tau_raw", "tau", "prob_deribit"]].isna().sum())

print("\nMatched by bet type:")
print(df_poly_matched["bet_type"].value_counts(dropna=False))

print("\nProbability summary:")
display(
    df_poly_matched[["prob_market", "prob_deribit", "mispricing", "tau_raw", "tau"]]
    .describe()
)

Polymarket-Deribit matched dataset
Input rows: 30,060
Rows with tau_raw <= 0 excluded: 88
Matched rows: 29,972
Markets matched: 3,431

Missing inputs:
spot             0
sigma            0
tau_raw          0
tau             88
prob_deribit    88
dtype: int64

Matched by bet type:
bet_type
above    10585
range     9064
reach     5810
dip       4513
Name: count, dtype: int64

Probability summary:


,prob_market,prob_deribit,mispricing,tau_raw,tau
count,29972.000000,29972.000000,29972.000000,29972.000000,29972.000000
mean,0.263815,0.229712,0.034103,0.058485,0.058485
std,0.320727,0.306729,0.097111,0.164380,0.164380
min,0.000500,0.000000,-0.998000,0.000455,0.000455
25%,0.021500,0.009797,-0.001888,0.007305,0.007305
50%,0.115000,0.102136,0.007953,0.012785,0.012785
75%,0.405000,0.285413,0.035755,0.018378,0.018378
max,0.999500,1.000000,0.762499,1.001370,1.001370


In [70]:
# ============================================================
# Define Polymarket matched analysis samples
# ============================================================

SHORT_HORIZON_DAYS = 30
VERY_SHORT_HORIZON_DAYS = 7

df_poly_matched["terminal_sample"] = df_poly_matched["bet_type"].isin(
    ["above", "range"]
)

df_poly_matched["path_dependent_sample"] = df_poly_matched["bet_type"].isin(
    ["reach", "dip"]
)

df_poly_matched["short_horizon_sample"] = (
    df_poly_matched["tau"] <= SHORT_HORIZON_DAYS / 365
)

df_poly_matched["very_short_horizon_sample"] = (
    df_poly_matched["tau"] <= VERY_SHORT_HORIZON_DAYS / 365
)

df_poly_matched_all = df_poly_matched.copy()

df_poly_matched_terminal = df_poly_matched[
    df_poly_matched["terminal_sample"]
].copy()

df_poly_matched_path_dep = df_poly_matched[
    df_poly_matched["path_dependent_sample"]
].copy()

df_poly_matched_terminal_short = df_poly_matched[
    df_poly_matched["terminal_sample"] &
    df_poly_matched["short_horizon_sample"]
].copy()

df_poly_matched_terminal_very_short = df_poly_matched[
    df_poly_matched["terminal_sample"] &
    df_poly_matched["very_short_horizon_sample"]
].copy()

print("=" * 70)
print("Polymarket matched sample definitions")
print("=" * 70)

print("All sample:")
print(f"  Rows: {len(df_poly_matched_all):,}")
print(f"  Markets: {df_poly_matched_all['market_id'].nunique():,}")

print("\nTerminal sample: above + range")
print(f"  Rows: {len(df_poly_matched_terminal):,}")
print(f"  Markets: {df_poly_matched_terminal['market_id'].nunique():,}")

print("\nPath-dependent sample: reach + dip")
print(f"  Rows: {len(df_poly_matched_path_dep):,}")
print(f"  Markets: {df_poly_matched_path_dep['market_id'].nunique():,}")

print(f"\nTerminal short-horizon sample: <= {SHORT_HORIZON_DAYS} days")
print(f"  Rows: {len(df_poly_matched_terminal_short):,}")
print(f"  Markets: {df_poly_matched_terminal_short['market_id'].nunique():,}")

print(f"\nTerminal very-short-horizon sample: <= {VERY_SHORT_HORIZON_DAYS} days")
print(f"  Rows: {len(df_poly_matched_terminal_very_short):,}")
print(f"  Markets: {df_poly_matched_terminal_very_short['market_id'].nunique():,}")

display(
    df_poly_matched
    .groupby("bet_type")
    .agg(
        n_obs=("market_id", "size"),
        n_markets=("market_id", "nunique"),
        avg_tau=("tau", "mean"),
        avg_prob_market=("prob_market", "mean"),
        avg_prob_deribit=("prob_deribit", "mean"),
        avg_mispricing=("mispricing", "mean"),
    )
    .round(4)
)

Polymarket matched sample definitions
All sample:
  Rows: 29,972
  Markets: 3,431

Terminal sample: above + range
  Rows: 19,649
  Markets: 2,847

Path-dependent sample: reach + dip
  Rows: 10,323
  Markets: 584

Terminal short-horizon sample: <= 30 days
  Rows: 19,649
  Markets: 2,847

Terminal very-short-horizon sample: <= 7 days
  Rows: 19,616
  Markets: 2,847


,n_obs,n_markets,avg_tau,avg_prob_market,avg_prob_deribit,avg_mispricing
bet_type,,,,,,
above,10585,1532,0.0102,0.4891,0.4814,0.0077
dip,4513,274,0.1297,0.1478,0.0810,0.0667
range,9064,1315,0.0101,0.1077,0.1004,0.0074
reach,5810,310,0.1667,0.1870,0.0884,0.0986


In [71]:
# ============================================================
# Polymarket matched diagnostics
# ============================================================

print("=" * 70)
print("Polymarket matched diagnostics")
print("=" * 70)

print("Rows with tau_raw <= 0 excluded:", (~df_poly_m["valid_time_to_expiry"]).sum())

print("\nMatched observations by bet type:")
print(df_poly_matched["bet_type"].value_counts(dropna=False))

print("\nTerminal sample probability summary:")
display(
    df_poly_matched_terminal[
        ["prob_market", "prob_deribit", "mispricing", "tau"]
    ].describe()
)

print("\nPath-dependent sample probability summary:")
display(
    df_poly_matched_path_dep[
        ["prob_market", "prob_deribit", "mispricing", "tau"]
    ].describe()
)

Polymarket matched diagnostics
Rows with tau_raw <= 0 excluded: 88

Matched observations by bet type:
bet_type
above    10585
range     9064
reach     5810
dip       4513
Name: count, dtype: int64

Terminal sample probability summary:


,prob_market,prob_deribit,mispricing,tau
count,19649.000000,19649.000000,19649.000000,19649.000000
mean,0.313195,0.305645,0.007551,0.010121
std,0.349895,0.342760,0.039818,0.005617
min,0.000500,0.000000,-0.499154,0.000799
25%,0.037500,0.046616,-0.009002,0.004680
50%,0.145000,0.140361,0.002772,0.010046
75%,0.560000,0.524205,0.018686,0.015525
max,0.999500,1.000000,0.491675,0.069863



Path-dependent sample probability summary:


,prob_market,prob_deribit,mispricing,tau
count,10323.000000,10323.000000,10323.000000,10323.000000
mean,0.169822,0.085179,0.084643,0.150542
std,0.228394,0.132962,0.143066,0.255871
min,0.000500,0.000000,-0.998000,0.000455
25%,0.008500,0.000038,0.006000,0.014268
50%,0.055000,0.016222,0.030254,0.045205
75%,0.260000,0.133023,0.126753,0.080022
max,0.999500,1.000000,0.762499,1.001370


In [75]:
# ============================================================
# Save temporally aligned Polymarket matched datasets
# ============================================================

poly_matched_path = (
    PROCESSED_DIR / "polymarket_deribit_matched.csv"
)

poly_terminal_path = (
    PROCESSED_DIR / "polymarket_deribit_matched_terminal.csv"
)

poly_path_dep_path = (
    PROCESSED_DIR / "polymarket_deribit_matched_path_dependent.csv"
)

poly_terminal_short_path = (
    PROCESSED_DIR / "polymarket_deribit_matched_terminal_short.csv"
)

poly_terminal_very_short_path = (
    PROCESSED_DIR /
    "polymarket_deribit_matched_terminal_very_short.csv"
)

df_poly_matched.to_csv(poly_matched_path, index=False)

df_poly_matched_terminal.to_csv(
    poly_terminal_path,
    index=False,
)

df_poly_matched_path_dep.to_csv(
    poly_path_dep_path,
    index=False,
)

df_poly_matched_terminal_short.to_csv(
    poly_terminal_short_path,
    index=False,
)

df_poly_matched_terminal_very_short.to_csv(
    poly_terminal_very_short_path,
    index=False,
)

print("=" * 70)
print("Temporally aligned Polymarket datasets saved")
print("=" * 70)

print("All matched:")
print(f"  Path: {poly_matched_path}")
print(f"  Rows: {len(df_poly_matched):,}")

print("\nTerminal:")
print(f"  Path: {poly_terminal_path}")
print(f"  Rows: {len(df_poly_matched_terminal):,}")
print(
    f"  Mean mispricing: "
    f"{df_poly_matched_terminal['mispricing'].mean():.6f}"
)

print("\nPath-dependent:")
print(f"  Path: {poly_path_dep_path}")
print(f"  Rows: {len(df_poly_matched_path_dep):,}")
print(
    f"  Mean mispricing: "
    f"{df_poly_matched_path_dep['mispricing'].mean():.6f}"
)

print("\nBenchmark timing columns saved:")
print(
    [
        "benchmark_source_date",
        "benchmark_available_at",
    ]
)

Temporally aligned Polymarket datasets saved
All matched:
  Path: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_deribit_matched.csv
  Rows: 29,972

Terminal:
  Path: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_deribit_matched_terminal.csv
  Rows: 19,649
  Mean mispricing: 0.007551

Path-dependent:
  Path: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_deribit_matched_path_dependent.csv
  Rows: 10,323
  Mean mispricing: 0.084643

Benchmark timing columns saved:
['benchmark_source_date', 'benchmark_available_at']


In [77]:
# ============================================================
# Construct timestamped Kalshi-Deribit matched dataset
# ============================================================

kalshi_benchmark_keep_cols = [
    "ticker",
    "observation_time",
    "spot",
    "dvol",
    "sigma",
    "spot_available_at",
    "spot_age_seconds",
    "benchmark_source_date",
    "benchmark_available_at",
    "dvol_age_hours",
]

df_kalshi_m = df_kalshi.merge(
    df_kalshi_benchmark[kalshi_benchmark_keep_cols],
    on=["ticker", "observation_time"],
    how="left",
    validate="one_to_one",
)

# Verify that all timestamped Kalshi prices received a benchmark
required_benchmark_cols = [
    "spot",
    "dvol",
    "sigma",
    "spot_available_at",
    "benchmark_source_date",
    "benchmark_available_at",
]

if df_kalshi_m[required_benchmark_cols].isna().any().any():
    missing_inputs = (
        df_kalshi_m[required_benchmark_cols]
        .isna()
        .sum()
    )

    raise ValueError(
        "Incomplete Kalshi intraday benchmark inputs:\n"
        f"{missing_inputs}"
    )

# Explicit no-look-ahead checks
if not (
    df_kalshi_m["spot_available_at"]
    <= df_kalshi_m["observation_time"]
).all():
    raise ValueError(
        "Look-ahead detected in Kalshi intraday spot."
    )

if not (
    df_kalshi_m["benchmark_available_at"]
    <= df_kalshi_m["observation_time"]
).all():
    raise ValueError(
        "Look-ahead detected in Kalshi DVOL benchmark."
    )

# Time from the actual first trade to contract close
df_kalshi_m["tau_raw"] = (
    (
        df_kalshi_m["close_time"]
        - df_kalshi_m["observation_time"]
    ).dt.total_seconds()
    / (365 * 24 * 60 * 60)
)

df_kalshi_m["valid_time_to_expiry"] = (
    df_kalshi_m["tau_raw"] > 0
)

df_kalshi_m["tau"] = np.nan
df_kalshi_m.loc[
    df_kalshi_m["valid_time_to_expiry"],
    "tau",
] = (
    df_kalshi_m.loc[
        df_kalshi_m["valid_time_to_expiry"],
        "tau_raw",
    ]
    .clip(lower=MIN_TAU)
)

df_kalshi_m["prob_market"] = (
    df_kalshi_m["prob_kalshi"]
)

df_kalshi_m["prob_deribit"] = np.nan

valid_probability_mask = (
    df_kalshi_m["valid_time_to_expiry"]
    & df_kalshi_m["prob_market"].between(
        0,
        1,
        inclusive="both",
    )
    & df_kalshi_m["spot"].gt(0)
    & df_kalshi_m["sigma"].gt(0)
)

df_kalshi_m.loc[
    valid_probability_mask,
    "prob_deribit",
] = df_kalshi_m.loc[
    valid_probability_mask
].apply(
    lambda row: compute_deribit_probability(
        row,
        source="kalshi",
    ),
    axis=1,
)

df_kalshi_m["mispricing"] = (
    df_kalshi_m["prob_market"]
    - df_kalshi_m["prob_deribit"]
)

df_kalshi_m["valid_match"] = (
    valid_probability_mask
    & df_kalshi_m["prob_deribit"].between(
        0,
        1,
        inclusive="both",
    )
    & df_kalshi_m["tau"].notna()
)

df_kalshi_matched = df_kalshi_m.loc[
    df_kalshi_m["valid_match"]
].copy()

df_kalshi_matched["tau_hours"] = (
    df_kalshi_matched["tau"] * 365 * 24
)

print("=" * 70)
print("Timestamped Kalshi-Deribit matched dataset")
print("=" * 70)

print(f"Input rows: {len(df_kalshi_m):,}")
print(
    "Rows with tau_raw <= 0 excluded:",
    (~df_kalshi_m["valid_time_to_expiry"]).sum(),
)
print(f"Matched rows: {len(df_kalshi_matched):,}")
print(
    "Events matched:",
    f"{df_kalshi_matched['event_ticker'].nunique():,}",
)
print(
    "Tickers matched:",
    f"{df_kalshi_matched['ticker'].nunique():,}",
)

print("\nMatched by strike type:")
print(
    df_kalshi_matched[
        "strike_type"
    ].value_counts(dropna=False)
)

print("\nMatched by price source:")
print(
    df_kalshi_matched[
        "price_source"
    ].value_counts(dropna=False)
)

print("\nMinutes after market open:")
display(
    df_kalshi_matched[
        "minutes_after_open"
    ].describe()
)

print("\nTau in minutes:")
display(
    (
        df_kalshi_matched["tau"]
        * 365 * 24 * 60
    ).describe()
)

print("\nBenchmark age:")
display(
    df_kalshi_matched[
        ["spot_age_seconds", "dvol_age_hours"]
    ].describe()
)

print("\nProbability summary:")
display(
    df_kalshi_matched[
        [
            "prob_market",
            "prob_deribit",
            "mispricing",
            "tau",
            "tau_hours",
        ]
    ].describe()
)

Timestamped Kalshi-Deribit matched dataset
Input rows: 3,135
Rows with tau_raw <= 0 excluded: 0
Matched rows: 3,135
Events matched: 485
Tickers matched: 3,135

Matched by strike type:
strike_type
between    3080
less         40
greater      15
Name: count, dtype: int64

Matched by price source:
price_source
first_timestamped_trade    3135
Name: count, dtype: int64

Minutes after market open:


count    3135.000000
mean        7.760279
std         7.512822
min         0.036169
25%         2.056121
50%         4.567508
75%        11.509700
max        29.960120
Name: minutes_after_open, dtype: float64


Tau in minutes:


count    3135.000000
mean       52.239721
std         7.512822
min        30.039880
25%        48.490300
50%        55.432492
75%        57.943879
max        59.963831
Name: tau, dtype: float64


Benchmark age:


,spot_age_seconds,dvol_age_hours
count,3135.000000,3135.000000
mean,29.120158,3.456292
std,17.919626,0.480972
min,0.010836,3.000603
25%,12.715739,3.053304
50%,29.570288,3.194515
75%,44.614176,4.029879
max,59.983175,4.499335



Probability summary:


,prob_market,prob_deribit,mispricing,tau,tau_hours
count,3135.000000,3135.000000,3135.000000,3135.000000,3135.000000
mean,0.184715,0.121641,0.063073,0.000099,0.870662
std,0.169293,0.078286,0.147670,0.000014,0.125214
min,0.010000,0.000000,-0.189085,0.000057,0.500665
25%,0.050000,0.060262,-0.013395,0.000092,0.808172
50%,0.150000,0.121626,0.019875,0.000105,0.923875
75%,0.270000,0.189150,0.097849,0.000110,0.965731
max,0.960000,0.999999,0.960000,0.000114,0.999397


In [79]:
# ============================================================
# Validate and save timestamped Kalshi matched datasets
# ============================================================

KALSHI_ROBUSTNESS_MAX_MINUTES = 15

# Variables expected by downstream notebooks
df_kalshi_matched["observation_date"] = (
    df_kalshi_matched["observation_time"].dt.normalize()
)

df_kalshi_matched["terminal_sample"] = True
df_kalshi_matched["path_dependent_sample"] = False

df_kalshi_matched_15m = df_kalshi_matched.loc[
    df_kalshi_matched["minutes_after_open"]
    <= KALSHI_ROBUSTNESS_MAX_MINUTES
].copy()

# ------------------------------------------------------------
# Critical validation
# ------------------------------------------------------------

assert df_kalshi_matched["ticker"].is_unique

assert df_kalshi_matched[
    [
        "event_ticker",
        "observation_time",
        "close_time",
        "prob_market",
        "prob_deribit",
        "mispricing",
        "spot",
        "sigma",
        "tau",
    ]
].notna().all().all()

assert df_kalshi_matched["prob_market"].between(
    0, 1, inclusive="both"
).all()

assert df_kalshi_matched["prob_deribit"].between(
    0, 1, inclusive="both"
).all()

assert (
    df_kalshi_matched["observation_time"]
    > df_kalshi_matched["open_time"]
).all()

assert (
    df_kalshi_matched["observation_time"]
    < df_kalshi_matched["close_time"]
).all()

assert (
    df_kalshi_matched["minutes_after_open"] <= 30
).all()

assert (
    df_kalshi_matched["spot_available_at"]
    <= df_kalshi_matched["observation_time"]
).all()

assert (
    df_kalshi_matched["benchmark_available_at"]
    <= df_kalshi_matched["observation_time"]
).all()

assert (
    df_kalshi_matched_15m["minutes_after_open"] <= 15
).all()

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

kalshi_matched_path = (
    PROCESSED_DIR / "kalshi_deribit_matched.csv"
)

kalshi_matched_15m_path = (
    PROCESSED_DIR / "kalshi_deribit_matched_15m.csv"
)

df_kalshi_matched.to_csv(
    kalshi_matched_path,
    index=False,
)

df_kalshi_matched_15m.to_csv(
    kalshi_matched_15m_path,
    index=False,
)

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print("=" * 70)
print("Timestamped Kalshi matched sample validation")
print("=" * 70)

print("Main sample: first trade within 30 minutes")
print(f"  Rows: {len(df_kalshi_matched):,}")
print(
    f"  Events: "
    f"{df_kalshi_matched['event_ticker'].nunique():,}"
)
print(
    f"  Mean mispricing: "
    f"{df_kalshi_matched['mispricing'].mean():.6f}"
)

print("\nRobustness sample: first trade within 15 minutes")
print(f"  Rows: {len(df_kalshi_matched_15m):,}")
print(
    f"  Events: "
    f"{df_kalshi_matched_15m['event_ticker'].nunique():,}"
)
print(
    f"  Mean mispricing: "
    f"{df_kalshi_matched_15m['mispricing'].mean():.6f}"
)

print("\nMain sample by strike type:")
display(
    df_kalshi_matched
    .groupby("strike_type")
    .agg(
        observations=("ticker", "size"),
        events=("event_ticker", "nunique"),
        mean_prob_market=("prob_market", "mean"),
        mean_prob_deribit=("prob_deribit", "mean"),
        mean_mispricing=("mispricing", "mean"),
        median_mispricing=("mispricing", "median"),
        mean_tau_minutes=(
            "tau",
            lambda x: x.mean() * 365 * 24 * 60,
        ),
    )
    .round(6)
)

print("\nSaved files:")
print(f"  Main Kalshi matched: {kalshi_matched_path}")
print(f"  Kalshi 15-minute robustness: {kalshi_matched_15m_path}")

print("\nKalshi matched validation passed.")

Timestamped Kalshi matched sample validation
Main sample: first trade within 30 minutes
  Rows: 3,135
  Events: 485
  Mean mispricing: 0.063073

Robustness sample: first trade within 15 minutes
  Rows: 2,554
  Events: 483
  Mean mispricing: 0.063855

Main sample by strike type:


,observations,events,mean_prob_market,mean_prob_deribit,mean_mispricing,median_mispricing,mean_tau_minutes
strike_type,,,,,,,
between,3080,484,0.185159,0.123358,0.061801,0.020485,52.287755
greater,15,15,0.178667,0.000000,0.178667,0.010000,48.043659
less,40,40,0.152750,0.035093,0.117657,0.010000,50.114559



Saved files:
  Main Kalshi matched: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/kalshi_deribit_matched.csv
  Kalshi 15-minute robustness: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/kalshi_deribit_matched_15m.csv

Kalshi matched validation passed.


In [81]:
# ============================================================
# Build unified cross-market matched dataset
# ============================================================

df_poly_unified = df_poly_matched.copy()

df_poly_unified["market"] = "polymarket"
df_poly_unified["contract_id"] = (
    "polymarket:" + df_poly_unified["market_id"].astype(str)
)
df_poly_unified["event_id_unified"] = df_poly_unified["event_id"]
df_poly_unified["observation_time"] = df_poly_unified["datetime"]
df_poly_unified["expiration_time"] = df_poly_unified["end_date"]
df_poly_unified["contract_type"] = df_poly_unified["bet_type"]
df_poly_unified["contract_subtitle"] = pd.NA
df_poly_unified["strike_lower"] = df_poly_unified["strike_lo"]
df_poly_unified["strike_upper"] = df_poly_unified["strike_hi"]
df_poly_unified["price_source"] = "price_history"
df_poly_unified["minutes_after_open"] = np.nan
df_poly_unified["first_trade_size"] = np.nan
df_poly_unified["spot_available_at"] = pd.NaT
df_poly_unified["spot_age_seconds"] = np.nan
df_poly_unified["dvol_age_hours"] = np.nan

df_kalshi_unified = df_kalshi_matched.copy()

df_kalshi_unified["market"] = "kalshi"
df_kalshi_unified["contract_id"] = (
    "kalshi:" + df_kalshi_unified["ticker"].astype(str)
)
df_kalshi_unified["event_id_unified"] = (
    df_kalshi_unified["event_ticker"]
)
df_kalshi_unified["date"] = (
    df_kalshi_unified["observation_time"].dt.normalize()
)
df_kalshi_unified["expiration_time"] = (
    df_kalshi_unified["close_time"]
)
df_kalshi_unified["contract_type"] = (
    df_kalshi_unified["strike_type"]
)
df_kalshi_unified["question"] = df_kalshi_unified.get(
    "title", pd.NA
)
df_kalshi_unified["contract_subtitle"] = df_kalshi_unified.get(
    "yes_sub_title", pd.NA
)
df_kalshi_unified["strike_lower"] = (
    df_kalshi_unified["floor_strike"]
)
df_kalshi_unified["strike_upper"] = (
    df_kalshi_unified["cap_strike"]
)
df_kalshi_unified["terminal_sample"] = True
df_kalshi_unified["path_dependent_sample"] = False
df_kalshi_unified["short_horizon_sample"] = True
df_kalshi_unified["very_short_horizon_sample"] = True

unified_cols = [
    "market", "contract_id", "event_id_unified",
    "date", "observation_time", "expiration_time",
    "contract_type", "question", "contract_subtitle",
    "strike_lower", "strike_upper",
    "prob_market", "prob_deribit", "mispricing",
    "spot", "sigma", "dvol", "tau", "tau_raw",
    "price_source", "benchmark_source_date",
    "benchmark_available_at", "spot_available_at",
    "spot_age_seconds", "dvol_age_hours",
    "minutes_after_open", "first_trade_size",
    "terminal_sample", "path_dependent_sample",
    "short_horizon_sample", "very_short_horizon_sample",
]

for frame in [df_poly_unified, df_kalshi_unified]:
    for col in unified_cols:
        if col not in frame.columns:
            frame[col] = pd.NA

df_unified = pd.concat(
    [
        df_poly_unified[unified_cols],
        df_kalshi_unified[unified_cols],
    ],
    ignore_index=True,
)

df_unified = (
    df_unified
    .sort_values(["observation_time", "market", "contract_id"])
    .reset_index(drop=True)
)

assert len(df_unified) == (
    len(df_poly_matched) + len(df_kalshi_matched)
)

assert df_unified[
    ["prob_market", "prob_deribit", "mispricing", "spot", "sigma", "tau"]
].notna().all().all()

assert not df_unified.duplicated(
    subset=["market", "contract_id", "observation_time"]
).any()

assert (
    df_unified["benchmark_available_at"]
    <= df_unified["observation_time"]
).all()

print("=" * 70)
print("Unified cross-market matched dataset")
print("=" * 70)

print(f"Rows: {len(df_unified):,}")
print(f"Contracts: {df_unified['contract_id'].nunique():,}")

print("\nRows by market:")
print(df_unified["market"].value_counts())

print("\nRows by market and contract type:")
display(
    df_unified.groupby(
        ["market", "contract_type"]
    ).size().to_frame("observations")
)

print("\nMispricing summary:")
display(
    df_unified.groupby("market").agg(
        observations=("contract_id", "size"),
        contracts=("contract_id", "nunique"),
        mean_prob_market=("prob_market", "mean"),
        mean_prob_deribit=("prob_deribit", "mean"),
        mean_mispricing=("mispricing", "mean"),
        median_mispricing=("mispricing", "median"),
    ).round(6)
)

Unified cross-market matched dataset
Rows: 33,107
Contracts: 6,566

Rows by market:
market
polymarket    29972
kalshi         3135
Name: count, dtype: int64

Rows by market and contract type:


observations
market     contract_type              
kalshi     between                3080
           greater                  15
           less                     40
polymarket above                 10585
           dip                    4513
           range                  9064
           reach                  5810


Mispricing summary:


,observations,contracts,mean_prob_market,mean_prob_deribit,mean_mispricing,median_mispricing
market,,,,,,
kalshi,3135,3135,0.184715,0.121641,0.063073,0.019875
polymarket,29972,3431,0.263815,0.229712,0.034103,0.007953


In [88]:
# ============================================================
# Save final matched datasets and construction summary
# ============================================================

unified_path = (
    PROCESSED_DIR /
    "unified_prediction_market_deribit_matched.csv"
)

unified_final_path = (
    FINAL_DIR /
    "unified_prediction_market_deribit_matched.csv"
)

df_unified.to_csv(unified_path, index=False)
df_unified.to_csv(unified_final_path, index=False)

def summarize_dataset(name, df, contract_col, date_col):
    return {
        "dataset": name,
        "rows": len(df),
        "contracts": df[contract_col].nunique(),
        "start_date": df[date_col].min(),
        "end_date": df[date_col].max(),
        "mean_prob_market": df["prob_market"].mean(),
        "mean_prob_deribit": df["prob_deribit"].mean(),
        "mean_mispricing": df["mispricing"].mean(),
    }

dataset_construction_summary = pd.DataFrame([
    summarize_dataset(
        "Polymarket-Deribit matched",
        df_poly_matched, "market_id", "date",
    ),
    summarize_dataset(
        "Polymarket-Deribit terminal",
        df_poly_matched_terminal, "market_id", "date",
    ),
    summarize_dataset(
        "Polymarket terminal very-short-horizon",
        df_poly_matched_terminal_very_short, "market_id", "date",
    ),
    summarize_dataset(
        "Polymarket path-dependent",
        df_poly_matched_path_dep, "market_id", "date",
    ),
    summarize_dataset(
        "Kalshi-Deribit timestamped 30-minute",
        df_kalshi_matched, "ticker", "observation_date",
    ),
    summarize_dataset(
        "Kalshi-Deribit timestamped 15-minute",
        df_kalshi_matched_15m, "ticker", "observation_date",
    ),
    summarize_dataset(
        "Unified matched dataset",
        df_unified, "contract_id", "date",
    ),
])

dataset_construction_summary_path = (
    TABLES_DIR / "dataset_construction_summary.csv"
)

dataset_construction_summary.to_csv(
    dataset_construction_summary_path,
    index=False,
)

display(
    dataset_construction_summary.round({
        "mean_prob_market": 6,
        "mean_prob_deribit": 6,
        "mean_mispricing": 6,
    })
)

,dataset,rows,contracts,start_date,end_date,mean_prob_market,mean_prob_deribit,mean_mispricing
0,Polymarket-Deribit matched,29972,3431,2024-03-09 00:00:00+00:00,2026-06-03 00:00:00+00:00,0.263815,0.229712,0.034103
1,Polymarket-Deribit terminal,19649,2847,2024-03-09 00:00:00+00:00,2026-06-03 00:00:00+00:00,0.313195,0.305645,0.007551
2,Polymarket terminal very-short-horizon,19616,2847,2024-03-09 00:00:00+00:00,2026-06-03 00:00:00+00:00,0.312867,0.305356,0.007511
3,Polymarket path-dependent,10323,584,2024-04-03 00:00:00+00:00,2026-06-01 00:00:00+00:00,0.169822,0.085179,0.084643
4,Kalshi-Deribit timestamped 30-minute,3135,3135,2025-02-04 00:00:00+00:00,2026-06-04 00:00:00+00:00,0.184715,0.121641,0.063073
5,Kalshi-Deribit timestamped 15-minute,2554,2554,2025-02-04 00:00:00+00:00,2026-06-04 00:00:00+00:00,0.194988,0.131133,0.063855
6,Unified matched dataset,33107,6566,2024-03-09 00:00:00+00:00,2026-06-04 00:00:00+00:00,0.256324,0.219478,0.036846


In [89]:
# ============================================================
# Final dataset construction export summary
# ============================================================

print("=" * 70)
print("Final dataset construction export summary")
print("=" * 70)

print("Input final datasets:")
print(f"  Polymarket: {polymarket_path}")
print(f"  Kalshi timestamped: {kalshi_path}")
print(f"  Deribit daily: {deribit_path}")
print(f"  Kalshi intraday benchmark: {kalshi_intraday_benchmark_path}")

print("\nOutput datasets:")
print(f"  Polymarket matched: {poly_matched_path}")
print(f"  Polymarket terminal: {poly_terminal_path}")
print(f"  Polymarket path-dependent: {poly_path_dep_path}")
print(f"  Polymarket terminal short: {poly_terminal_short_path}")
print(
    "  Polymarket terminal very short:",
    poly_terminal_very_short_path,
)
print(f"  Kalshi timestamped matched: {kalshi_matched_path}")
print(f"  Kalshi 15-minute robustness: {kalshi_matched_15m_path}")
print(f"  Unified processed: {unified_path}")
print(f"  Unified final: {unified_final_path}")

print("\nSummary table:")
print(f"  {dataset_construction_summary_path}")

print("\nFinal sample:")
print(f"  Unified rows: {len(df_unified):,}")
print(
    f"  Unified contracts: "
    f"{df_unified['contract_id'].nunique():,}"
)
print(f"  Start: {df_unified['date'].min()}")
print(f"  End:   {df_unified['date'].max()}")

print("\nQuality checks:")
print(
    "  Missing core values:",
    df_unified[
        ["prob_market", "prob_deribit", "mispricing", "spot", "sigma", "tau"]
    ].isna().sum().sum(),
)
print(
    "  Invalid market probabilities:",
    (~df_unified["prob_market"].between(0, 1)).sum(),
)
print(
    "  Invalid benchmark probabilities:",
    (~df_unified["prob_deribit"].between(0, 1)).sum(),
)
print(
    "  Duplicate market-contract-time rows:",
    df_unified.duplicated(
        subset=["market", "contract_id", "observation_time"]
    ).sum(),
)
print(
    "  Benchmark look-ahead violations:",
    (
        df_unified["benchmark_available_at"]
        > df_unified["observation_time"]
    ).sum(),
)

print("\n04_dataset_construction.ipynb completed successfully.")

Final dataset construction export summary
Input final datasets:
  Polymarket: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final/polymarket_btc_prices_analysis.csv
  Kalshi timestamped: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final/kalshi_kxbtc_first_trades_timestamped_30m.csv
  Deribit daily: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final/deribit_btc_daily_inputs.csv
  Kalshi intraday benchmark: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final/deribit_kalshi_intraday_inputs.csv

Output datasets:
  Polymarket matched: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_deribit_matched.csv
  Polymarket terminal: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_deribit_matched_terminal.csv
  Polymarket path-dependent: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_deribit_matched_path_dependent.csv
  Polymarket termin